# Entrega 1: Tabela Gold de Frentes com Membros
Cria a tabela `ouro.frentes_membros_gold` unindo frentes e membros com dados de partido, UF e legislatura.

In [0]:
# Registrar views da prata para usar em SQL
spark.sql("SELECT * FROM prata.frentes").createOrReplaceTempView("prata_frentes_view")
spark.sql("SELECT * FROM prata.frentes_membros").createOrReplaceTempView("prata_membros_view")
print("Views criadas.")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ouro;

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.frentes_membros_gold
USING DELTA
COMMENT 'Camada Ouro - Frentes com membros, partido, UF e legislatura'
AS
SELECT
    f.id_frente,
    f.titulo AS nome_frente,
    f.id_legislatura,
    f.nome_coordenador,
    f.partido_coordenador,
    f.uf_coordenador,
    m.id_deputado,
    m.nome AS nome_deputado,
    m.sigla_partido,
    m.sigla_uf,
    m.cargo
FROM prata_frentes_view f
JOIN prata_membros_view m ON f.id_frente = m.id_frente;

# Entrega 2: Índice de Herfindahl
Mede a diversidade partidária de cada frente. Quanto mais próximo de 0, mais diversa.


In [0]:
%sql
CREATE OR REPLACE TABLE ouro.herfindahl_frentes
USING DELTA
COMMENT 'Camada Ouro - Indice de Herfindahl por frente (diversidade partidaria)'
AS
WITH proporcoes AS (
    SELECT 
        id_frente,
        nome_frente,
        id_legislatura,
        sigla_partido,
        COUNT(*) AS qtd_membros,
        SUM(COUNT(*)) OVER (PARTITION BY id_frente) AS total_membros
    FROM ouro.frentes_membros_gold
    WHERE sigla_partido IS NOT NULL
    GROUP BY id_frente, nome_frente, id_legislatura, sigla_partido
),
herfindahl AS (
    SELECT
        id_frente,
        nome_frente,
        id_legislatura,
        SUM(POWER(qtd_membros * 1.0 / total_membros, 2)) AS indice_herfindahl,
        COUNT(DISTINCT sigla_partido) AS qtd_partidos,
        MAX(total_membros) AS total_membros
    FROM proporcoes
    GROUP BY id_frente, nome_frente, id_legislatura
)
SELECT * FROM herfindahl;

In [0]:
%sql
-- Top 10 frentes mais diversas (menor indice)
SELECT 
    nome_frente,
    id_legislatura,
    ROUND(indice_herfindahl, 4) AS herfindahl,
    qtd_partidos,
    total_membros
FROM ouro.herfindahl_frentes
ORDER BY indice_herfindahl ASC
LIMIT 10;

In [0]:
%sql
-- Top 10 frentes mais concentradas (maior indice)
SELECT 
    nome_frente,
    id_legislatura,
    ROUND(indice_herfindahl, 4) AS herfindahl,
    qtd_partidos,
    total_membros
FROM ouro.herfindahl_frentes
ORDER BY indice_herfindahl DESC
LIMIT 10;

# Entrega 3: Deputados com Mais Frentes
Lista os deputados que participam do maior número de frentes parlamentares, com exemplos de temas.

In [0]:
%sql

CREATE OR REPLACE TABLE ouro.deputados_multifrentes
USING DELTA
COMMENT 'Camada Ouro - Deputados com mais frentes e seus temas'
AS
WITH lista_frentes AS (
    SELECT 
        id_deputado,
        nome_deputado,
        sigla_partido,
        sigla_uf,
        COLLECT_LIST(nome_frente) AS frentes_array,
        COUNT(DISTINCT id_frente) AS qtd_frentes,
        COUNT(DISTINCT id_legislatura) AS qtd_legislaturas
    FROM ouro.frentes_membros_gold
    WHERE sigla_partido IS NOT NULL
    GROUP BY id_deputado, nome_deputado, sigla_partido, sigla_uf
)
SELECT 
    id_deputado,
    nome_deputado,
    sigla_partido,
    sigla_uf,
    qtd_frentes,
    qtd_legislaturas,
    CONCAT_WS(' | ',
        get(frentes_array, 0),
        get(frentes_array, 1),
        get(frentes_array, 2),
        get(frentes_array, 3),
        get(frentes_array, 4)
    ) AS exemplos_frentes
FROM lista_frentes
ORDER BY qtd_frentes DESC;

In [0]:
%sql
-- Top 10 deputados com mais frentes
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    qtd_frentes,
    exemplos_frentes
FROM ouro.deputados_multifrentes
ORDER BY qtd_frentes DESC
LIMIT 10;

In [0]:
%sql
-- Media de frentes por partido (partidos com deputados mais engajados)
SELECT 
    sigla_partido,
    ROUND(AVG(qtd_frentes), 1) AS media_frentes,
    MAX(qtd_frentes) AS maximo_frentes,
    COUNT(*) AS qtd_deputados
FROM ouro.deputados_multifrentes
GROUP BY sigla_partido
HAVING COUNT(*) >= 5
ORDER BY media_frentes DESC
LIMIT 10;

# Entrega 5: Evolução de Frentes por Tema
Categoriza frentes por tema (Saúde, Educação, Agropecuária, etc.) e contabiliza por legislatura.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.evolucao_frentes_tema
USING DELTA
COMMENT 'Camada Ouro - Evolucao do numero de frentes por tema entre legislaturas'
AS
SELECT 
    id_legislatura,
    CASE 
        WHEN LOWER(titulo) LIKE '%saúde%' OR LOWER(titulo) LIKE '%saude%' OR LOWER(titulo) LIKE '%hospital%' 
            OR LOWER(titulo) LIKE '%medic%' OR LOWER(titulo) LIKE '%câncer%' OR LOWER(titulo) LIKE '%cancer%'
            THEN 'Saude'
        WHEN LOWER(titulo) LIKE '%educação%' OR LOWER(titulo) LIKE '%educacao%' OR LOWER(titulo) LIKE '%escola%'
            OR LOWER(titulo) LIKE '%universidade%' OR LOWER(titulo) LIKE '%estudante%' OR LOWER(titulo) LIKE '%ensino%'
            OR LOWER(titulo) LIKE '%instituto federal%'
            THEN 'Educacao'
        WHEN LOWER(titulo) LIKE '%agro%' OR LOWER(titulo) LIKE '%rural%' OR LOWER(titulo) LIKE '%agric%'
            OR LOWER(titulo) LIKE '%pecuária%' OR LOWER(titulo) LIKE '%pecuaria%' OR LOWER(titulo) LIKE '%café%'
            OR LOWER(titulo) LIKE '%cafe%' OR LOWER(titulo) LIKE '%leite%' OR LOWER(titulo) LIKE '%erva-mate%'
            OR LOWER(titulo) LIKE '%floresta%'
            THEN 'Agropecuaria'
        WHEN LOWER(titulo) LIKE '%indústria%' OR LOWER(titulo) LIKE '%industria%' OR LOWER(titulo) LIKE '%comércio%'
            OR LOWER(titulo) LIKE '%comercio%' OR LOWER(titulo) LIKE '%serviço%' OR LOWER(titulo) LIKE '%servico%'
            OR LOWER(titulo) LIKE '%empreendedor%' OR LOWER(titulo) LIKE '%micro%' OR LOWER(titulo) LIKE '%empresa%'
            OR LOWER(titulo) LIKE '%negócio%' OR LOWER(titulo) LIKE '%negocio%' OR LOWER(titulo) LIKE '%franqueada%'
            OR LOWER(titulo) LIKE '%banco%'
            THEN 'Economia e Negocios'
        WHEN LOWER(titulo) LIKE '%direito%' OR LOWER(titulo) LIKE '%humano%' OR LOWER(titulo) LIKE '%criança%'
            OR LOWER(titulo) LIKE '%crianca%' OR LOWER(titulo) LIKE '%idoso%' OR LOWER(titulo) LIKE '%mulher%'
            OR LOWER(titulo) LIKE '%autis%' OR LOWER(titulo) LIKE '%deficiente%' OR LOWER(titulo) LIKE '%inclusão%'
            OR LOWER(titulo) LIKE '%inclusao%' OR LOWER(titulo) LIKE '%pessoa%' OR LOWER(titulo) LIKE '%social%'
            OR LOWER(titulo) LIKE '%suicídio%' OR LOWER(titulo) LIKE '%suicidio%'
            THEN 'Direitos e Sociais'
        WHEN LOWER(titulo) LIKE '%ambient%' OR LOWER(titulo) LIKE '%sustent%' OR LOWER(titulo) LIKE '%clima%'
            OR LOWER(titulo) LIKE '%energia%' OR LOWER(titulo) LIKE '%amazônia%' OR LOWER(titulo) LIKE '%amazonia%'
            OR LOWER(titulo) LIKE '%reciclagem%' OR LOWER(titulo) LIKE '%hidro%' OR LOWER(titulo) LIKE '%elétrica%'
            OR LOWER(titulo) LIKE '%eletrica%'
            THEN 'Meio Ambiente e Energia'
        WHEN LOWER(titulo) LIKE '%segurança%' OR LOWER(titulo) LIKE '%seguranca%' 
            OR LOWER(titulo) LIKE '%corrupção%' OR LOWER(titulo) LIKE '%corrupcao%'
            OR LOWER(titulo) LIKE '%violência%' OR LOWER(titulo) LIKE '%violencia%'
            OR LOWER(titulo) LIKE '%penitenciário%' OR LOWER(titulo) LIKE '%penitenciario%'
            OR LOWER(titulo) LIKE '%jogos%' OR LOWER(titulo) LIKE '%arma%'
            THEN 'Seguranca e Justica'
        WHEN LOWER(titulo) LIKE '%cultura%' OR LOWER(titulo) LIKE '%esporte%' OR LOWER(titulo) LIKE '%turismo%'
            OR LOWER(titulo) LIKE '%samba%' OR LOWER(titulo) LIKE '%carnaval%' OR LOWER(titulo) LIKE '%arte%'
            THEN 'Cultura e Esporte'
        WHEN LOWER(titulo) LIKE '%tecnologia%' OR LOWER(titulo) LIKE '%digital%' OR LOWER(titulo) LIKE '%inovação%'
            OR LOWER(titulo) LIKE '%inovacao%' OR LOWER(titulo) LIKE '%inteligência%' OR LOWER(titulo) LIKE '%inteligencia%'
            OR LOWER(titulo) LIKE '%ciência%' OR LOWER(titulo) LIKE '%ciencia%'
            THEN 'Tecnologia e Inovacao'
        WHEN LOWER(titulo) LIKE '%transporte%' OR LOWER(titulo) LIKE '%rodovia%' OR LOWER(titulo) LIKE '%br-%'
            OR LOWER(titulo) LIKE '%mobilidade%' OR LOWER(titulo) LIKE '%estrada%' OR LOWER(titulo) LIKE '%ferrovia%'
            OR LOWER(titulo) LIKE '%porto%' OR LOWER(titulo) LIKE '%aeroporto%'
            THEN 'Infraestrutura e Transporte'
        ELSE 'Outros'
    END AS tema,
    COUNT(*) AS qtd_frentes
FROM prata.frentes
GROUP BY id_legislatura, tema
ORDER BY id_legislatura, qtd_frentes DESC;

In [0]:
%sql
SELECT 
    tema,
    SUM(qtd_frentes) AS total_frentes,
    COUNT(DISTINCT id_legislatura) AS legislaturas
FROM ouro.evolucao_frentes_tema
GROUP BY tema
ORDER BY total_frentes DESC;

# Tabela Auxiliar: Classificação Ideológica
Classifica frentes como Progressista ou Conservadora com base em palavras-chave no título.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.classificacao_ideologica_frentes
USING DELTA
COMMENT 'Classificacao ideologica das frentes baseada em temas e palavras-chave'
AS
SELECT 
    id_frente,
    titulo,
    id_legislatura,
    CASE 
        -- Frentes de tendencia progressista
        WHEN LOWER(titulo) LIKE '%direitos humanos%' 
            OR LOWER(titulo) LIKE '%igualdade%'
            OR LOWER(titulo) LIKE '%feminista%'
            OR LOWER(titulo) LIKE '%lgbt%'
            OR LOWER(titulo) LIKE '%diversidade%'
            OR LOWER(titulo) LIKE '%indígena%' OR LOWER(titulo) LIKE '%indigena%'
            OR LOWER(titulo) LIKE '%quilombola%'
            OR LOWER(titulo) LIKE '%racial%'
            OR LOWER(titulo) LIKE '%gênero%' OR LOWER(titulo) LIKE '%genero%'
            OR LOWER(titulo) LIKE '%inclusão%' OR LOWER(titulo) LIKE '%inclusao%'
            OR LOWER(titulo) LIKE '%reciclagem%'
            OR LOWER(titulo) LIKE '%sustent%'
            OR LOWER(titulo) LIKE '%amazônia%' OR LOWER(titulo) LIKE '%amazonia%'
            OR LOWER(titulo) LIKE '%meio ambiente%'
            OR LOWER(titulo) LIKE '%clima%'
            THEN 'Progressista'
            
        -- Frentes de tendencia conservadora
        WHEN LOWER(titulo) LIKE '%evangélica%' OR LOWER(titulo) LIKE '%evangelica%'
            OR LOWER(titulo) LIKE '%contra o aborto%'
            OR LOWER(titulo) LIKE '%em defesa da vida%'
            OR LOWER(titulo) LIKE '%conservadora%'
            OR LOWER(titulo) LIKE '%sem doutrinação%' OR LOWER(titulo) LIKE '%sem doutrinacao%'
            OR LOWER(titulo) LIKE '%família%' OR LOWER(titulo) LIKE '%familia%'
            OR LOWER(titulo) LIKE '%cristã%' OR LOWER(titulo) LIKE '%crista%'
            OR LOWER(titulo) LIKE '%armas%' OR LOWER(titulo) LIKE '%atirador%' OR LOWER(titulo) LIKE '%caçador%'
            OR LOWER(titulo) LIKE '%liberdade de expressão%' OR LOWER(titulo) LIKE '%liberdade de expressao%'
            OR LOWER(titulo) LIKE '%agropecuária%' OR LOWER(titulo) LIKE '%agropecuaria%'
            OR LOWER(titulo) LIKE '%contra a corrupção%' OR LOWER(titulo) LIKE '%contra a corrupcao%'
            OR LOWER(titulo) LIKE '%combate à corrupção%' OR LOWER(titulo) LIKE '%combate a corrupcao%'
            OR LOWER(titulo) LIKE '%sem jogos%'
            THEN 'Conservadora'
            
        ELSE 'Nao classificada'
    END AS orientacao_ideologica
FROM prata.frentes;

In [0]:
%sql
SELECT 
    orientacao_ideologica,
    COUNT(*) AS qtd_frentes,
    COUNT(DISTINCT id_legislatura) AS legislaturas
FROM ouro.classificacao_ideologica_frentes
GROUP BY orientacao_ideologica
ORDER BY qtd_frentes DESC;

# Entrega 4: Sobreposição Ideológica
Cruza deputados que estão simultaneamente em frentes progressistas e conservadoras, gerando pares de frentes opostas.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.sobreposicao_ideologica
USING DELTA
COMMENT 'Camada Ouro - Pares de frentes ideologicamente opostas por deputado'
AS
WITH frentes_classificadas AS (
    SELECT id_frente, orientacao_ideologica
    FROM ouro.classificacao_ideologica_frentes
    WHERE orientacao_ideologica IN ('Progressista', 'Conservadora')
),
membros_progressistas AS (
    SELECT DISTINCT 
        m.id_deputado,
        m.nome_deputado,
        m.sigla_partido,
        m.sigla_uf,
        m.id_frente,
        m.nome_frente
    FROM ouro.frentes_membros_gold m
    JOIN frentes_classificadas f ON m.id_frente = f.id_frente
    WHERE f.orientacao_ideologica = 'Progressista'
),
membros_conservadores AS (
    SELECT DISTINCT 
        m.id_deputado,
        m.nome_deputado,
        m.sigla_partido,
        m.sigla_uf,
        m.id_frente,
        m.nome_frente
    FROM ouro.frentes_membros_gold m
    JOIN frentes_classificadas f ON m.id_frente = f.id_frente
    WHERE f.orientacao_ideologica = 'Conservadora'
)
SELECT 
    p.id_deputado,
    p.nome_deputado,
    p.sigla_partido,
    p.sigla_uf,
    p.nome_frente AS frente_progressista,
    c.nome_frente AS frente_conservadora,
    CONCAT(p.nome_frente, '  |||  ', c.nome_frente) AS par_oposicao
FROM membros_progressistas p
JOIN membros_conservadores c ON p.id_deputado = c.id_deputado
ORDER BY p.nome_deputado, p.nome_frente, c.nome_frente

In [0]:
%sql
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    frente_progressista,
    frente_conservadora
FROM ouro.sobreposicao_ideologica
ORDER BY nome_deputado
LIMIT 20